### Procesamiento de Lenguaje Natural I
# **Desafío 1**

#### Alumna: Federica Pavese - a2321


In [1]:
%pip install numpy scikit-learn

### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  4703, 10870,  4333])

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.




---

**RESOLUCIÓN DEL DESAFÍO**

---



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

In [24]:
import pandas as pd
from IPython.display import display, HTML

# Fija semilla
np.random.seed(42)

# Tomamos 5 documentos al azar del conjunto de entrenamiento
random_docs = np.random.choice(X_train.shape[0], size=5, replace=False)

random_docs

array([7492, 3546, 5582, 4793, 3813])

In [25]:
for idx in random_docs:

    print("=" * 100)
    print(f"DOCUMENTO ORIGINAL - índice {idx}")
    print(f"Clase real: {newsgroups_train.target_names[y_train[idx]]}")
    print("-" * 100)
    print(newsgroups_train.data[idx][:1000])

    # Similaridad coseno
    cossim = cosine_similarity(X_train[idx], X_train)[0]

    # Top 5 más similares (excluyendo el mismo documento)
    mostsim = np.argsort(cossim)[::-1][1:6]

    resultados = []

    print("\n5 DOCUMENTOS MÁS SIMILARES:\n")

    for ranking, sim_idx in enumerate(mostsim, start=1):

        clase = newsgroups_train.target_names[y_train[sim_idx]]
        similitud = cossim[sim_idx]

        resultados.append({
            "Ranking": ranking,
            "Índice": sim_idx,
            "Clase": clase,
            "Similaridad coseno": round(similitud, 4)
        })

        print("-" * 100)
        print(f"{ranking}) Índice: {sim_idx}")
        print(f"Clase: {clase}")
        print(f"Similaridad coseno: {similitud:.4f}")
        print("\n")
        print(newsgroups_train.data[sim_idx][:700])


    # Información del documento original
    df_original = pd.DataFrame([{
        "Índice": idx,
        "Clase": newsgroups_train.target_names[y_train[idx]]
    }])

    display(HTML("""
    <br><br>
    <b style='font-size:16px;'>DOCUMENTO ORIGINAL</b>
    <hr style='width:500px; margin-left:0;'>
    """))

    display(df_original)

    # Tabla resumen documentos más similares
    df_resultados = pd.DataFrame(resultados)

    display(HTML("""
    <br><br>
    <b style='font-size:16px;'>DOCUMENTOS MÁS SIMILARES</b>
    <hr style='width:500px; margin-left:0;'>
    """))

    display(df_resultados)

    print("\n\n")

DOCUMENTO ORIGINAL - índice 7492
Clase real: comp.sys.mac.hardware
----------------------------------------------------------------------------------------------------
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" 

5 DOCUMENTOS MÁS SIMILARES:

----------------------------------------------------------------------------------------------------
1) Índice: 10935
Clase: comp.sys.mac.hardware
Similaridad coseno: 0.6665


Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyone out there who has found a really good deal on a Centris 650
send me the price.  I don't want to know where, unless it is mail order or
areound cleveland, Ohio.  Also, should I buy now or wait for the Power PC.

Thanks.
BoB
reply via post or 

,Índice,Clase
0,7492,comp.sys.mac.hardware


,Ranking,Índice,Clase,Similaridad coseno
0,1,10935,comp.sys.mac.hardware,0.6665
1,2,7258,comp.sys.ibm.pc.hardware,0.3476
2,3,4971,comp.sys.mac.hardware,0.1799
3,4,4303,misc.forsale,0.1547
4,5,645,comp.sys.mac.hardware,0.1414





DOCUMENTO ORIGINAL - índice 3546
Clase real: comp.os.ms-windows.misc
----------------------------------------------------------------------------------------------------


     Don't bother if you have CPBackup or Fastback.  They all offer options 
not available in the stripped-down MS version (FROM CPS!).  Examples - no 
proprietary format (to save space), probably no direct DMA access, and no 
tape drive!

5 DOCUMENTOS MÁS SIMILARES:

----------------------------------------------------------------------------------------------------
1) Índice: 5665
Clase: comp.sys.ibm.pc.hardware
Similaridad coseno: 0.2040



By initiating a DMA xfer.  :)

Seriously, busmastering adapter have their own DMA ability, they don't use
the motherboards on-board DMA(which is *MUCH* slower).

ISA has no bus arbitration, so if two busmastering cards in 1 ISA system
try to do DMA xfers on the same DMA channel the system will lock or 
crash.(I forget)

Their are 8 DMA channels in an ISA system. 0-7. 0-3 are

,Índice,Clase
0,3546,comp.os.ms-windows.misc


...

,Índice,Clase
0,4793,talk.politics.guns


,Ranking,Índice,Clase,Similaridad coseno
0,1,6894,talk.politics.guns,0.2364
1,2,5856,sci.crypt,0.2363
2,3,4271,talk.politics.misc,0.2328
3,4,3141,talk.politics.guns,0.2295
4,5,10836,alt.atheism,0.2291





DOCUMENTO ORIGINAL - índice 3813
Clase real: rec.sport.hockey
----------------------------------------------------------------------------------------------------

Doesn't it also have the Statue of Liberty on it or is that Richter's Mask?

The back actually has a Bee followed by a Z to represent the Beezer. It 
also has something that looks like the three interconnecting circles from
the Led Zepplin 4 album cover. Is that what it is supposed to be? and if
it is does anybody know why he would put it there? Ali?


John
"The official Language of Golf is Profanity"




5 DOCUMENTOS MÁS SIMILARES:

----------------------------------------------------------------------------------------------------
1) Índice: 10836
Clase: alt.atheism
Similaridad coseno: 0.2514


Archive-name: atheism/faq
Alt-atheism-archive-name: faq
Last-modified: 5 April 1993
Version: 1.1

                    Alt.Atheism Frequently-Asked Questions

This file contains responses to articles which occur repeatedly in
alt.

,Índice,Clase
0,3813,rec.sport.hockey


,Ranking,Índice,Clase,Similaridad coseno
0,1,10836,alt.atheism,0.2514
1,2,759,soc.religion.christian,0.2480
2,3,913,alt.atheism,0.2410
3,4,5826,soc.religion.christian,0.2409
4,5,5856,sci.crypt,0.2329


**Conclusiones - Punto 1**

En este punto se seleccionaron 5 documentos aleatorios del conjunto de entrenamiento y se calculó la similaridad coseno entre cada uno y el resto de los documentos utilizando la representación TF-IDF.

En general, los documentos más similares suelen pertenecer a la misma categoría o a categorías relacionadas. Esto muestra que TF-IDF logra representar bastante bien el contenido de los textos a partir de las palabras más importantes de cada documento.

También se observó que los documentos con mayor similaridad comparten vocabulario técnico o términos específicos de cada tema. Por ejemplo, en categorías relacionadas con computación, deportes o ciencia aparecen muchas palabras características que ayudan a agrupar correctamente los textos.

Sin embargo, la similaridad no siempre coincide exactamente con la clase verdadera. En algunos casos aparecieron documentos de categorías distintas con valores relativamente altos de similaridad coseno. Por ejemplo, se encontró una similaridad cercana a 0.23 entre documentos de las clases `rec.sport.hockey` y `sci.crypt`.

Esto ocurre porque TF-IDF trabaja únicamente con frecuencia de palabras y no comprende el significado semántico del texto. Por lo tanto, documentos de temas diferentes pueden compartir ciertos términos, nombres propios o formas de escritura que aumentan la similaridad.

En conclusión, la similaridad coseno sobre vectores TF-IDF resulta útil para recuperar documentos relacionados temáticamente, aunque presenta limitaciones cuando distintas categorías comparten parte del vocabulario.

---



**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

In [26]:
# Vectorizamos el conjunto de test utilizando el TF-IDF ajustado en train
X_test = tfidfvect.transform(newsgroups_test.data)

# Lista para guardar las predicciones
y_pred_proto = []

# Recorremos todos los documentos de test
for i in range(X_test.shape[0]):

    # Similaridad del documento test i contra TODOS los de train
    sim = cosine_similarity(X_test[i], X_train)[0]

    # Índice del documento train más similar
    best_idx = np.argmax(sim)

    # Clase del documento más similar
    pred_class = y_train[best_idx]

    y_pred_proto.append(pred_class)

# Convertimos a array
y_pred_proto = np.array(y_pred_proto)

# Desempeño global del clasificador por prototipos comparando clases reales vs clases predichas.
# average='macro' promedia el F1-score de todas las clases otorgando el mismo peso a cada una.
f1_proto = f1_score(y_test, y_pred_proto, average='macro')

print(f"F1-score macro (clasificación por prototipos): {f1_proto:.4f}")

F1-score macro (clasificación por prototipos): 0.5050


In [27]:
# Ejemplos de predicción

resultados_pred = []

for i in range(5):

    # Similaridad contra train
    sim = cosine_similarity(X_test[i], X_train)[0]

    # Documento train más similar
    best_idx = np.argmax(sim)

    print("=" * 100)

    print(f"Documento TEST {i}")
    print(f"Clase real: {newsgroups_test.target_names[y_test[i]]}")
    print(f"Clase predicha: {newsgroups_train.target_names[y_pred_proto[i]]}")
    print(f"Similaridad máxima: {sim[best_idx]:.4f}")

    print("\nDocumento TRAIN más similar:\n")
    print(newsgroups_train.data[best_idx][:700])

    resultados_pred.append({
        "Documento TEST": i,
        "Clase real": newsgroups_test.target_names[y_test[i]],
        "Clase predicha": newsgroups_train.target_names[y_pred_proto[i]],
        "Similaridad máxima": round(sim[best_idx], 4)
    })

    print("\n\n")

# Tabla resumen
df_pred = pd.DataFrame(resultados_pred)

display(HTML("""
<b style='font-size:16px;'>TABLA RESUMEN DE PREDICCIONES</b>
<hr style='width:750px; margin-left:0;'>
"""))

display(df_pred)

Documento TEST 0
Clase real: rec.autos
Clase predicha: alt.atheism
Similaridad máxima: 0.2596

Documento TRAIN más similar:

The recent rise of nostalgia in this group, combined with the
  incredible level of utter bullshit, has prompted me to comb
  through my archives and pull out some of "The Best of Alt.Atheism"
  for your reading pleasure.  I'll post a couple of these a day
  unless group concensus demands that I stop, or I run out of good
  material.

  I haven't been particularly careful in the past about saving
  attributions.  I think the following comes from John A. Johnson,
  but someone correct me if I'm wrong.  This is probably the longest
  of my entire collection.

________________________________________________________


                                  So that the
                               



Documento TEST 1
Clase real: comp.windows.x
Clase predicha: talk.religion.misc
Similaridad máxima: 0.2342

Documento TRAIN más similar:


If I have a habit that I really w

,Documento TEST,Clase real,Clase predicha,Similaridad máxima
0,0,rec.autos,alt.atheism,0.2596
1,1,comp.windows.x,talk.religion.misc,0.2342
2,2,alt.atheism,talk.politics.mideast,0.3391
3,3,talk.politics.mideast,talk.politics.mideast,0.4253
4,4,talk.religion.misc,alt.atheism,0.2761


**Conclusiones - Punto 2**

En este punto se implementó un clasificador por prototipos basado en similaridad coseno utilizando los vectores TF-IDF de los documentos. Para cada documento del conjunto de test se buscó el documento más similar del conjunto de entrenamiento y se asignó su misma clase.

Los resultados muestran que el método logra capturar cierta relación temática entre documentos, aunque el desempeño es bastante limitado. En los ejemplos analizados, solo uno de los cinco documentos fue clasificado correctamente (`talk.politics.mideast`), mientras que el resto fue asignado a categorías incorrectas.

También se observa que las similaridades máximas obtenidas son relativamente bajas, en torno a valores entre 0.23 y 0.42. Esto indica que muchos documentos no tienen una correspondencia extremadamente cercana dentro del conjunto de entrenamiento.

En varios casos las clases predichas pertenecen a temas distintos de la clase real, por ejemplo un documento de `rec.autos` fue clasificado como `alt.atheism`. Esto ocurre porque el método se basa únicamente en similitud de vocabulario y no comprende realmente el significado del texto.

---



**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

In [28]:
# Lista de experimentos

experimentos = [
    {
        "modelo": "MultinomialNB baseline",
        "descripcion": "Baseline utilizando TfidfVectorizer y MultinomialNB con parámetros default",
        "vectorizer": TfidfVectorizer(),
        "modelo_nb": MultinomialNB()
    },
    {
        "modelo": "MultinomialNB + stopwords",
        "descripcion": "Se agregan stopwords para eliminar palabras frecuentes del inglés",
        "vectorizer": TfidfVectorizer(
            stop_words='english'
        ),
        "modelo_nb": MultinomialNB()
    },
    {
        "modelo": "MultinomialNB + min/max_df",
        "descripcion": """
            Se ajusta la frecuencia mínima y máxima de términos
            min_df=3 elimina palabras poco frecuentes
            max_df=0.8 elimina palabras demasiado frecuentes
            """,
        "vectorizer": TfidfVectorizer(
            stop_words='english',
            min_df=3,
            max_df=0.8
        ),
        "modelo_nb": MultinomialNB()
    },
    {
        "modelo": "MultinomialNB + sublinear_tf",
        "descripcion": "Se agrega sublinear_tf=True para suavizar el peso de palabras muy repetidas",
        "vectorizer": TfidfVectorizer(
            stop_words='english',
            min_df=3,
            max_df=0.8,
            sublinear_tf=True
        ),
        "modelo_nb": MultinomialNB()
    },
    {
        "modelo": "MultinomialNB optimizado",
        "descripcion": """
            Se ajusta alpha=0.1 para evitar probabilidades iguales a cero,
            que podrían descartar completamente una clasey darle más peso a las frecuencias observadas
            """,
        "vectorizer": TfidfVectorizer(
            stop_words='english',
            min_df=3,
            max_df=0.8,
            sublinear_tf=True
        ),
        "modelo_nb": MultinomialNB(alpha=0.1)
    },
    {
        "modelo": "ComplementNB optimizado",
        "descripcion": """
            Se prueba ComplementNB, una variante de Naive Bayes
            que suele funcionar bien con clases desbalanceadas
            """,
        "vectorizer": TfidfVectorizer(
            stop_words='english',
            min_df=3,
            max_df=0.8,
            sublinear_tf=True
        ),
        "modelo_nb": ComplementNB(alpha=0.1)
    },
    {
        "modelo": "ComplementNB alpha=0.01",
        "descripcion": """
            Se reduce alpha a 0.01 para disminuir el suavizado
            y priorizar las frecuencias reales observadas
            """,
        "vectorizer": TfidfVectorizer(
            stop_words='english',
            min_df=3,
            max_df=0.8,
            sublinear_tf=True
        ),
        "modelo_nb": ComplementNB(alpha=0.01)
    }
]

In [29]:
# Lista para guardar resultados
resultados_nb = []

# Recorremos los experimentos
for exp in experimentos:

    # Vectorizador
    vect = exp["vectorizer"]

    # Matrices documento-término
    X_train_nb = vect.fit_transform(newsgroups_train.data)
    X_test_nb = vect.transform(newsgroups_test.data)

    # Targets
    y_train_nb = newsgroups_train.target
    y_test_nb = newsgroups_test.target

    # Modelo
    model = exp["modelo_nb"]

    # Entrenamiento
    model.fit(X_train_nb, y_train_nb)

    # Predicción
    y_pred_nb = model.predict(X_test_nb)

    # Evaluación
    f1_nb = f1_score(y_test_nb, y_pred_nb, average='macro')

    # Guardamos resultado
    resultados_nb.append({
        "MODELO": exp["modelo"],
        "DESCRIPCION": exp["descripcion"].strip().replace("\n", "<br>"),
        "F1-SCORE MACRO": round(f1_nb, 4)
    })

# Tabla final
df_resultados_nb = pd.DataFrame(resultados_nb)

# Mostrar descripción completa
pd.set_option('display.max_colwidth', None)

display(HTML("""
<b style='font-size:16px;'>RESULTADOS MODELOS NAÏVE BAYES</b>
<hr style='width:1200px; margin-left:0;'>
"""))

display(
    df_resultados_nb.style
    .hide(axis="index")
    .set_properties(
        subset=["MODELO", "DESCRIPCION"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [("text-align", "left")]
        }
    ])
)

MODELO,DESCRIPCION,F1-SCORE MACRO
MultinomialNB baseline,Baseline utilizando TfidfVectorizer y MultinomialNB con parámetros default,0.585400
MultinomialNB + stopwords,Se agregan stopwords para eliminar palabras frecuentes del inglés,0.646800
MultinomialNB + min/max_df,Se ajusta la frecuencia mínima y máxima de términos min_df=3 elimina palabras poco frecuentes max_df=0.8 elimina palabras demasiado frecuentes,0.653000
MultinomialNB + sublinear_tf,Se agrega sublinear_tf=True para suavizar el peso de palabras muy repetidas,0.646500
MultinomialNB optimizado,"Se ajusta alpha=0.1 para evitar probabilidades iguales a cero, que podrían descartar completamente una clasey darle más peso a las frecuencias observadas",0.678500
ComplementNB optimizado,"Se prueba ComplementNB, una variante de Naive Bayes que suele funcionar bien con clases desbalanceadas",0.682100
ComplementNB alpha=0.01,Se reduce alpha a 0.01 para disminuir el suavizado y priorizar las frecuencias reales observadas,0.673400


**Conclusiones - Punto 3**

En este punto se probaron distintos modelos de Naïve Bayes y diferentes configuraciones del vectorizador TF-IDF para intentar mejorar el desempeño de clasificación.

El modelo baseline obtuvo un F1-score macro de 0.5854, que sirve como referencia inicial. A partir de ahí, casi todos los cambios realizados ayudaron a mejorar el resultado.

Agregar stopwords produjo una mejora importante, pasando a un F1-score de 0.6468. Esto tiene sentido porque se eliminaron palabras muy frecuentes del inglés que no aportan demasiada información sobre la temática de los documentos.

Luego, ajustar `min_df` y `max_df` mejoró un poco más el desempeño, llegando a 0.6530. Filtrar palabras demasiado raras o demasiado frecuentes ayudó a reducir ruido en el vocabulario.

En cambio, usar `sublinear_tf=True` no produjo una mejora en este caso y el resultado quedó prácticamente igual al experimento anterior.

Los mejores resultados aparecieron al ajustar el parámetro `alpha`. Tanto `MultinomialNB(alpha=0.1)` como `ComplementNB(alpha=0.1)` mejoraron bastante respecto al baseline. El mejor modelo fue `ComplementNB optimizado`, con un F1-score macro de 0.6821.

Finalmente, bajar alpha a 0.01 no ayudó y el desempeño cayó un poco. Esto muestra que reducir demasiado el suavizado también puede perjudicar al modelo.

En general, se vio que el preprocesamiento del texto y el ajuste de hiperparámetros tienen bastante impacto en tareas de clasificación NLP, incluso utilizando modelos relativamente simples como Naïve Bayes.

---



**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [30]:
# Transponemos la matriz documento-término
X_words = X_train.T

print(f"Shape original X_train: {X_train.shape}")
print(f"Shape transpuesta X_words: {X_words.shape}")

Shape original X_train: (11314, 101631)
Shape transpuesta X_words: (101631, 11314)


In [31]:
# Similaridad entre palabras

# Elegimos 5 palabras manualmente
palabras = ["cat", "business", "children", "medicine", "computer"]

resultados_palabras = []

for palabra in palabras:

    # Verificamos que la palabra exista en el vocabulario
    if palabra not in tfidfvect.vocabulary_:
        print(f"La palabra '{palabra}' no está en el vocabulario.")
        continue

    # Índice de la palabra en el vocabulario
    word_idx = tfidfvect.vocabulary_[palabra]

    # Vector de la palabra
    word_vector = X_words[word_idx]

    # Similaridad con todas las palabras
    sim_words = cosine_similarity(word_vector, X_words)[0]

    # Top 5 palabras más similares, excluyendo la misma palabra
    mostsim_words = np.argsort(sim_words)[::-1][1:6]

    for rank, sim_idx in enumerate(mostsim_words, start=1):
        resultados_palabras.append({
            "Palabra original": palabra,
            "Ranking": rank,
            "Palabra similar": idx2word[sim_idx],
            "Similaridad coseno": round(sim_words[sim_idx], 4)
        })

df_palabras = pd.DataFrame(resultados_palabras)

display(HTML("""
<b style='font-size:16px;'>SIMILARIDAD ENTRE PALABRAS</b>
<hr style='width:800px; margin-left:0;'>
"""))

display(df_palabras)

,Palabra original,Ranking,Palabra similar,Similaridad coseno
0,cat,1,decorations,0.4763
1,cat,2,uptight,0.3937
2,cat,3,literate,0.3329
3,cat,4,jburnside,0.3201
4,cat,5,hahahha,0.2914
5,business,1,uwo,0.2419
6,business,2,regina,0.2415
7,business,3,s4lawren,0.2415
8,business,4,gophers,0.2415
9,business,5,reunion,0.2411


**Conclusiones - Punto 4 - Primera versión**

En esta primera prueba se transpuso la matriz documento-término original para analizar la similaridad entre palabras. La idea fue representar cada palabra según los documentos en los que aparece y calcular similaridad coseno entre esos vectores.

Los resultados muestran que el método funciona técnicamente, pero aparecen muchas palabras poco interpretables entre las más similares. Por ejemplo, para `cat` aparecen términos como `decorations`, `uptight`, `jburnside` o `hahahha`, que no parecen tener una relación semántica clara.

Algo similar ocurre con palabras como `business`, `medicine` o `computer`, donde aparecen términos raros, errores de tipeo, nombres propios o palabras muy específicas del corpus.

Esto pasa porque el vocabulario original contiene mucho ruido: palabras poco frecuentes, términos mal escritos, usuarios, nombres propios y expresiones informales típicas de mensajes de newsgroups.

Por lo tanto, esta primera versión permite ver una limitación importante: la similaridad entre palabras no depende solamente del significado, sino también de la coocurrencia en los mismos documentos. Si dos palabras aparecen en pocos documentos parecidos, pueden quedar como similares aunque semánticamente no lo sean.

Como mejora, se propone reconstruir la matriz TF-IDF usando un vectorizador más filtrado, eliminando stopwords y palabras demasiado raras o demasiado frecuentes.

In [32]:
# Similaridad entre palabras - versión filtrada


# Vectorizador más filtrado
tfidfvect_words = TfidfVectorizer(
    stop_words='english',
    min_df=5,
    max_df=0.8
)

# Nueva matriz documento-término
X_train_words = tfidfvect_words.fit_transform(newsgroups_train.data)

# Matriz término-documento
X_words = X_train_words.T

# Diccionario índice -> palabra
idx2word_words = {
    v: k for k, v in tfidfvect_words.vocabulary_.items()
}

print(f"Shape X_train_words: {X_train_words.shape}")
print(f"Shape X_words: {X_words.shape}")

Shape X_train_words: (11314, 17797)
Shape X_words: (17797, 11314)


In [33]:
# Palabras elegidas manualmente
palabras = ["cat", "business", "children", "medicine", "computer"]

resultados_palabras_filtrado = []

for palabra in palabras:

    # Verificamos que exista en el vocabulario
    if palabra not in tfidfvect_words.vocabulary_:
        print(f"La palabra '{palabra}' no está en el vocabulario.")
        continue

    # Índice de la palabra
    word_idx = tfidfvect_words.vocabulary_[palabra]

    # Vector de la palabra
    word_vector = X_words[word_idx]

    # Similaridad con todas las palabras
    sim_words = cosine_similarity(word_vector, X_words)[0]

    # Top 5 similares
    mostsim_words = np.argsort(sim_words)[::-1][1:6]

    for rank, sim_idx in enumerate(mostsim_words, start=1):

        resultados_palabras_filtrado.append({
            "Palabra original": palabra,
            "Ranking": rank,
            "Palabra similar": idx2word_words[sim_idx],
            "Similaridad coseno": round(sim_words[sim_idx], 4)
        })

# Tabla
df_palabras_filtrado = pd.DataFrame(resultados_palabras_filtrado)

display(HTML("""
<b style='font-size:16px;'>SIMILARIDAD ENTRE PALABRAS - VERSIÓN FILTRADA</b>
<hr style='width:900px; margin-left:0;'>
"""))

display(df_palabras_filtrado)

,Palabra original,Ranking,Palabra similar,Similaridad coseno
0,cat,1,ate,0.2930
1,cat,2,advantages,0.2471
2,cat,3,panther,0.2375
3,cat,4,christmas,0.2261
4,cat,5,tree,0.2180
5,business,1,uwo,0.2361
6,business,2,bureau,0.1574
7,business,3,yee,0.1531
8,business,4,badge,0.1317
9,business,5,supplying,0.1300


**Conclusiones - Punto 4 - Versión filtrada**

En esta segunda versión se volvió a calcular la similaridad entre palabras, pero usando un vectorizador más filtrado. Se eliminaron stopwords y también palabras demasiado poco frecuentes, con el objetivo de reducir ruido en el vocabulario.

Los resultados mejoraron parcialmente, aunque todavía aparecen asociaciones algo difíciles de interpretar. Por ejemplo, para `children` aparecen palabras bastante coherentes como `parents` y `parent`, lo cual tiene sentido porque suelen aparecer en contextos parecidos.

También en `medicine` aparece `placebo`, que sí tiene una relación temática clara. En el caso de `computer`, aparecen términos como `drive` y `hackers`, que también pueden vincularse con temas tecnológicos.

Sin embargo, todavía hay resultados raros. Por ejemplo, para `business` aparecen palabras como `uwo`, `yee` o `badge`, y para `cat` aparecen términos como `advantages` o `christmas`, que no tienen una relación semántica tan clara.

Esto muestra que el filtrado mejora un poco la calidad de las asociaciones, pero no elimina completamente el ruido del corpus. Además, esta técnica no aprende el significado real de las palabras, sino que mide si aparecen en documentos similares.

En conclusión, transponer la matriz documento-término permite construir una representación simple de palabras, pero sus resultados dependen mucho del vocabulario y del preprocesamiento.